# 01 — Exploração dos dados (Olist, avaliações de pedidos)

**Objetivo desta etapa:** entender o dataset *antes* de rotular ou treinar qualquer coisa.
Queremos responder: quantas avaliações têm texto? Como as notas se distribuem (classes desbalanceadas)? Há duplicatas que poderiam causar vazamento de dados? Os temas (entrega, qualidade, atendimento, preço, embalagem) aparecem nos textos?

**Fonte:** Brazilian E-Commerce Public Dataset by Olist (Kaggle) — licença **CC BY-NC-SA 4.0** (uso não comercial, com atribuição e compartilhamento pela mesma licença).

**Como preparar:** baixe o zip no Kaggle, extraia e coloque `olist_order_reviews_dataset.csv` em `data/raw/olist/`. Rode este notebook a partir da pasta `notebooks/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

SEED = 42  # semente fixa: mesmos sorteios sempre (reprodutibilidade)

RAW_DIR = Path("../data/raw/olist")
REVIEWS_CSV = RAW_DIR / "olist_order_reviews_dataset.csv"
assert REVIEWS_CSV.exists(), f"Arquivo não encontrado: {REVIEWS_CSV.resolve()}"

df = pd.read_csv(REVIEWS_CSV)
print("Formato (linhas, colunas):", df.shape)
df.head()

## 1. Estrutura e valores ausentes
Cada linha é uma avaliação de um pedido. O texto está em `review_comment_message` e é **opcional**: muitas pessoas dão só a nota. Só as linhas com texto servem para um classificador de texto.

In [ ]:
df.info()
print()
print("Proporção de valores ausentes por coluna:")
print(df.isna().mean().round(3))

df["tem_comentario"] = df["review_comment_message"].notna()
print(f"\nCom comentário: {df['tem_comentario'].sum()} de {len(df)} ({df['tem_comentario'].mean():.1%})")

## 2. Distribuição das notas
Se as classes forem muito desiguais, a **acurácia engana** (um modelo que sempre chuta a classe maior parece bom). Por isso usaremos F1 macro. Vamos ver o tamanho do desequilíbrio, primeiro em todas as avaliações e depois só nas com texto.

In [ ]:
todas = df["review_score"].value_counts().sort_index()
com = df[df["tem_comentario"]].copy()
so_texto = com["review_score"].value_counts().sort_index()

tabela = pd.DataFrame({
    "todas_n": todas,
    "todas_%": (todas / todas.sum() * 100).round(1),
    "com_texto_n": so_texto,
    "com_texto_%": (so_texto / so_texto.sum() * 100).round(1),
})
print(tabela)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), sharey=False)
todas.plot(kind="bar", ax=ax[0], title="Todas as avaliações")
so_texto.plot(kind="bar", ax=ax[1], title="Somente com texto")
for a in ax:
    a.set_xlabel("nota")
plt.tight_layout()
plt.show()

## 3. Prévia do sentimento (derivado da nota)
Regra **provisória** só para enxergar o desequilíbrio: 1–2 → negativo, 3 → neutro, 4–5 → positivo. A decisão final e o código oficial ficam para a etapa 2 (`src/prepare_data.py`).

In [ ]:
mapa = {1: "negativo", 2: "negativo", 3: "neutro", 4: "positivo", 5: "positivo"}
com["sentimento_previa"] = com["review_score"].map(mapa)
print(com["sentimento_previa"].value_counts())
print()
print(com["sentimento_previa"].value_counts(normalize=True).round(3))

## 4. Tamanho dos textos
O tamanho em palavras ajuda a definir o `max_length` do tokenizador mais adiante: curto demais corta informação, longo demais gasta memória da GPU (T4 do Colab).

In [ ]:
com["n_palavras"] = com["review_comment_message"].str.split().str.len()
com["n_caracteres"] = com["review_comment_message"].str.len()
print(com[["n_palavras", "n_caracteres"]].describe(percentiles=[.5, .9, .95, .99]).round(1))

com["n_palavras"].clip(upper=120).hist(bins=40, figsize=(7, 3.5))
plt.title("Palavras por avaliação (cortado em 120)")
plt.xlabel("palavras")
plt.show()

## 5. Duplicatas e risco de vazamento
**Vazamento de dados** acontece quando informação do teste "vaza" para o treino. Dois riscos aqui: (a) o **mesmo pedido** com várias avaliações; (b) **textos idênticos**. Se um cair no treino e outro no teste, o modelo "decora" e a métrica fica falsamente alta. Vamos medir antes de decidir como dividir.

In [ ]:
print("review_id duplicado:", df["review_id"].duplicated().sum())
print("order_id com mais de 1 avaliação:", df["order_id"].duplicated().sum())

textos = com["review_comment_message"].str.strip().str.lower()
print("textos idênticos entre os comentados:", textos.duplicated().sum())
print("\nTextos mais repetidos:")
print(textos.value_counts().head(8))

## 6. Lendo exemplos de verdade
Estatística não substitui ler. Vamos olhar alguns textos por nota, com semente fixa para o sorteio ser o mesmo sempre.

In [ ]:
for nota in [1, 3, 5]:
    print(f"\n=== nota {nota} ===")
    amostra = com[com["review_score"] == nota]["review_comment_message"].sample(5, random_state=SEED)
    for t in amostra:
        print("-", t.replace("\n", " ")[:250])

## 7. Os temas aparecem? (checagem rápida por palavras-chave)
Isto **não** é a rotulagem de tema (essa será feita na etapa 3 com um LLM e revisão manual). É só uma sondagem: se um tema quase não aparece, talvez precisemos ajustar a lista de temas. Atenção: palavras como "caixa" são ambíguas, então os números são aproximados.

In [ ]:
temas = {
    "entrega/atraso": r"entreg|atras|prazo|demor|chegou",
    "qualidade do produto": r"qualidade|defeito|quebrad|funcion|original",
    "atendimento": r"atendiment|suporte|vendedor|resposta|contato",
    "preço": r"pre[çc]o|caro|barato|valor|custo",
    "embalagem": r"embalag|caixa|lacrad|amassad",
}
txt = com["review_comment_message"].str.lower()
contagem = {tema: int(txt.str.contains(padrao, regex=True).sum()) for tema, padrao in temas.items()}
resumo = pd.Series(contagem, name="n_textos").to_frame()
resumo["% dos textos"] = (resumo["n_textos"] / len(com) * 100).round(1)
resumo

## 8. O que anotar para a próxima etapa
Preencha depois de rodar (isso vira material do README e do model card):
- Quantas avaliações têm texto? Qual a proporção de cada sentimento?
- O desequilíbrio é grande? O que isso implica para a métrica?
- Há duplicatas/pedidos repetidos? Como vamos evitar vazamento na divisão?
- Qual percentil de palavras cobre ~95% dos textos (ajuda a escolher `max_length`)?
- Algum tema quase não aparece?